# Esketamine plus oral antidepressant vs placebo spray plus oral antidepressant

**Illustrative model using published TA854 inputs; not a replication.**

Adults with treatment-resistant depression. NHS perspective. Costs and QALYs discounted at 3.5% (NICE reference case). Cycle length 4 weeks. Horizon 5 years (65 cycles), the company's original horizon in TA854. Comparator is placebo nasal spray plus a newly started oral antidepressant, the comparison used in the company submission.

States: depressed, responding/remitted, dead. Transitions: depressed to responding, responding back to depressed, any state to dead.

Two contested assumptions are parameters, not fixed facts:

- treatment duration: drug and administration costs stop after `treatment_cycles`
- waning: after stopping, the intervention relapse rate drifts back toward the comparator

List price. No confidential patient access scheme. Do not quote these ICERs as NICE or company results.


## Input log

| Input | Value used | Source |
|---|---|---|
| Population | Adults with TRD, start age 46 | TA854 scope; SUSTAIN-1 mean age 46.3 (Daly et al. 2019) |
| Comparator | Placebo spray + newly started oral antidepressant | Company submission comparison, TRANSFORM-2 design |
| Perspective, discount, cycle, horizon | NHS, 3.5%, 4 weeks, 5 years | NICE reference case; company original horizon (TA854 s3.24) |
| Induction response | 0.693 vs 0.520 | TRANSFORM-2 day 28, Popova et al. 2019; TA854 committee papers (69% vs 52%) |
| Induction remission | 0.525 vs 0.310, used only to weight the collapsed utility | Popova et al. 2019; TA854 committee papers (53% vs 31%) |
| 4-week relapse | 0.067787 vs 0.127454 | Derived from SUSTAIN-1 stable remitters, 26.7% vs 45.3% relapsed, median exposure 17.7 weeks. `p = 1-(1-F)^(4/17.7)` |
| Company recurrence rates, scenario only | 0.0243 vs 0.0356 | TA854 committee papers, company model, SUSTAIN-1 arms |
| Re-response after cycle 1 | 0.08 both arms | Illustrative. Below TRANSFORM-2, in the range of STAR*D later steps used in TA854. Not a copied transition |
| Utility, depressed | 0.417 | TRANSFORM-2 baseline, EQ-5D-5L mapped to 3L, TA854 committee papers |
| Utility, response / remission | 0.764 / 0.866; collapsed base case 0.841 | TA854 committee papers. Weight is a simplification |
| Health-state cost per cycle | £980 depressed, £164 responding | TA854 committee papers, document 1-2, UK chart review n=295 |
| Esketamine list price | £489 per 84 mg session | BNF April 2022, quoted in TA854 section 2.3. 56 mg is £326, used in sensitivity |
| Sessions per cycle | 8, then 4, then 2 | SmPC: twice weekly, then weekly, then every 1-2 weeks. Fortnightly from cycle 3 |
| Admin per session | £25 base; £150 committee-style | Constructed from the 6:1 vs 1:1 dispute (TA854 s3.29, s3.35). Not a published unit cost |
| Background mortality | annual qx 0.0016, constant, both arms | Approximation to ONS England national life tables 2018-2020, age 46, SUSTAIN-1 sex mix. No treatment effect on mortality |
| Oral antidepressant | £4 per cycle, both arms | Illustrative generic BNF cost. Cancels in the ICER |


In [1]:
from dataclasses import asdict
import model as m

base = m.Inputs()
result = m.incremental(base)

print("Horizon: {:.0f} years, {:.0f} cycles".format(base.horizon_years, base.horizon_years * m.CYCLES_PER_YEAR))
print("Discount rate: {:.1%}".format(base.discount_rate))
print()
print("Esketamine   cost £{:,.0f}   QALYs {:.3f}   LYs {:.3f}".format(
    result["esketamine"]["cost"], result["esketamine"]["qaly"], result["esketamine"]["ly"]))
print("Comparator   cost £{:,.0f}   QALYs {:.3f}   LYs {:.3f}".format(
    result["comparator"]["cost"], result["comparator"]["qaly"], result["comparator"]["ly"]))
print()
print("Incremental cost  £{:,.0f}".format(result["inc_cost"]))
print("Incremental QALYs {:.4f}".format(result["inc_qaly"]))
print("ICER              £{:,.0f} per QALY".format(result["icer"]))
print()
print("Discounted cost split, esketamine arm:")
e = result["esketamine"]
print("  drug £{:,.0f}   admin £{:,.0f}   health state £{:,.0f}   oral AD £{:,.0f}".format(
    e["drug_cost"], e["admin_cost"], e["state_cost"], e["oral_cost"]))
print()
for line in m.checks(result):
    print(line)


Horizon: 5 years, 65 cycles
Discount rate: 3.5%

Esketamine   cost £46,404   QALYs 2.770   LYs 4.576
Comparator   cost £39,451   QALYs 2.671   LYs 4.576

Incremental cost  £6,954
Incremental QALYs 0.0992
ICER              £70,068 per QALY

Discounted cost split, esketamine arm:
  drug £8,978   admin £459   health state £36,730   oral AD £238

esketamine: final state sum 1.000000; discounted QALYs 2.770 vs life-years 4.576
comparator: final state sum 1.000000; discounted QALYs 2.671 vs life-years 4.576


## State trace

The cohort starts depressed. Cycle 0 is the induction cycle (TRANSFORM-2 response). From cycle 1, only people who responded at induction still incur esketamine costs, and only until cycle 9. Later response from the induction-failure pool is not retreated. After cycle 9 the relapse rate drifts toward the comparator.


In [2]:
trace = result["esketamine"]["trace"][["cycle", "depressed", "responding", "dead", "on_treatment", "p_relapse", "effect"]]
print(trace.head(12).round(4).to_string(index=False))
print("...")
print(trace.iloc[[12, 25, 64]].round(4).to_string(index=False))
m.plot_trace(result, "outputs/trace.png")


 cycle  depressed  responding   dead  on_treatment  p_relapse  effect
     0     1.0000      0.0000 0.0000        1.0000     0.0678  1.0000
     1     0.3070      0.6929 0.0001        0.6929     0.0678  1.0000
     2     0.3293      0.6704 0.0002        0.6459     0.0678  1.0000
     3     0.3484      0.6512 0.0004        0.6058     0.0678  1.0000
     4     0.3646      0.6349 0.0005        0.5716     0.0678  1.0000
     5     0.3784      0.6209 0.0006        0.5425     0.0678  1.0000
     6     0.3902      0.6091 0.0007        0.5176     0.0678  1.0000
     7     0.4002      0.5989 0.0009        0.4965     0.0678  1.0000
     8     0.4088      0.5903 0.0010        0.4784     0.0678  1.0000
     9     0.4160      0.5829 0.0011        0.0000     0.0767  0.8500
    10     0.4274      0.5714 0.0012        0.0000     0.0843  0.7225
    11     0.4414      0.5573 0.0014        0.0000     0.0908  0.6141
...
 cycle  depressed  responding   dead  on_treatment  p_relapse  effect
    12     0.456

## Named scenarios

Company-style: costs stop at 9 cycles, but the effect does not wane, administration is the cheap 6:1 figure, and relapse uses the company's 2.43% and 3.56% recurrence probabilities.

Company-style with trial relapse: same maintained effect, but relapse is the SUSTAIN-1 conversion rather than the company recurrence rate.

Committee-style: effect wanes by half each cycle after stopping, administration is a 1:1 figure (£150), and the esketamine relapse probability is raised from 0.068 to 0.10 to stand in for the selection-bias concern. That 0.10 is a scenario knob, not an estimate.


In [3]:
company = m.Inputs(
    treatment_cycles=9, waning_per_cycle=0.0, admin_per_session=25.0,
    p_relapse_esk=0.0243, p_relapse_comp=0.0356,
)
company_trial = m.Inputs(
    treatment_cycles=9, waning_per_cycle=0.0, admin_per_session=25.0,
)
committee = m.Inputs(
    treatment_cycles=9, waning_per_cycle=0.50, admin_per_session=150.0,
    p_relapse_esk=0.100,
)
scenarios = m.scenario_table({
    "base_illustrative": base,
    "company_style": company,
    "company_style_trial_relapse": company_trial,
    "committee_style": committee,
})
show = scenarios[["scenario", "waning_per_cycle", "admin_per_session", "p_relapse_esk", "p_relapse_comp", "inc_cost", "inc_qaly", "icer"]].copy()
show["inc_cost"] = show["inc_cost"].round(0)
show["inc_qaly"] = show["inc_qaly"].round(3)
show["icer"] = show["icer"].round(0)
print(show.to_string(index=False))
print()
print("Crosses £20,000-£30,000?")
for row in scenarios.itertuples(index=False):
    inside = 20000 <= row.icer <= 30000
    below = row.icer < 20000
    print(f"  {row.scenario}: £{row.icer:,.0f}  {'inside' if inside else 'below' if below else 'above'} the range")


                   scenario  waning_per_cycle  admin_per_session  p_relapse_esk  p_relapse_comp  inc_cost  inc_qaly     icer
          base_illustrative              0.15               25.0       0.067787        0.127454    6954.0     0.099  70068.0
              company_style              0.00               25.0       0.024300        0.035600    5818.0     0.169  34404.0
company_style_trial_relapse              0.00               25.0       0.067787        0.127454    1642.0     0.312   5272.0
            committee_style              0.50              150.0       0.100000        0.127454   10033.0     0.048 207199.0

Crosses £20,000-£30,000?
  base_illustrative: £70,068  above the range
  company_style: £34,404  above the range
  company_style_trial_relapse: £5,272  below the range
  committee_style: £207,199  above the range


## One-way sensitivity

Ranges are the ones that map onto the dispute: duration 4 to 18 cycles, waning 0 to 0.50, 56 mg versus 84 mg list price, admin £25 versus £150, and the committee-paper utilities at either end of the collapsed state.


In [4]:
spec = [
    ("Treatment duration (cycles)", "treatment_cycles", 4, 18),
    ("Waning per cycle", "waning_per_cycle", 0.0, 0.50),
    ("Drug cost per session (£)", "price_per_session", 326.0, 489.0),
    ("Admin cost per session (£)", "admin_per_session", 25.0, 150.0),
    ("Utility, depressed", "u_depressed", 0.350, 0.550),
    ("Utility, responding", "u_responding", 0.764, 0.866),
]
owsa = m.one_way(base, spec)
spans = m.tornado_spans(owsa)
shown = spans.copy()
shown[["icer_low", "icer_high", "span"]] = shown[["icer_low", "icer_high", "span"]].round(0)
print(shown.to_string(index=False))
m.plot_tornado(spans, result["icer"], "outputs/tornado.png")
print()
print("Largest swing: {} (£{:,.0f})".format(spans.iloc[0]["parameter"], spans.iloc[0]["span"]))


                  parameter  icer_low  icer_high    span  value_low  value_high
           Waning per cycle    5272.0    95570.0 90299.0      0.000       0.500
         Utility, depressed   60507.0   102092.0 41585.0      0.350       0.550
  Drug cost per session (£)   39914.0    70068.0 30154.0    326.000     489.000
 Admin cost per session (£)   70068.0    93192.0 23124.0     25.000     150.000
        Utility, responding   85616.0    66167.0 19450.0      0.764       0.866
Treatment duration (cycles)   67986.0    67380.0   605.0      4.000      18.000


Largest swing: Waning per cycle (£90,299)


## Why this is not TA854

Published company results were about £7,389 to £10,790 per QALY (incremental cost about £2,700 to £3,400, incremental QALYs about 0.32 to 0.37). The ERG case was about £62,000. The committee's most plausible list-price range was £64,554 to £73,158.

This base case lands near that list-price range, at about £70,000. That is not agreement. The model has three states rather than five, no subsequent-line matrix, no recovery state, no carer disutility, an 84 mg list price on every session, and a relapse rate converted from the SUSTAIN-1 cumulative endpoint rather than the company's recurrence probabilities. The match is the right order of magnitude for a list-price run, not a replication.

The input that moves the ICER is waning, not duration. A maintained effect crosses below £20,000. Fast waning, a 1:1 administration cost, and a higher esketamine relapse rate stay far above £30,000. That is the dispute the committee was actually having.
